# Merge attack and normal datasets

## Loading and cleaning

In [2]:
# Make project helpers available from a kernel started in this folder or the repo root.
from pathlib import Path
import sys
PROJECT_ROOT = next(root for root in (Path.cwd(), *Path.cwd().parents)
                    if (root / "preprocessing").is_dir() and (root / "scenario_reconstructor").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

In [3]:
DATA_ROOT = PROJECT_ROOT / "data" / "cAPTure"
NORMAL_DATA_ROOT = DATA_ROOT / "normal"
ATTACK_DATA_ROOT = DATA_ROOT / "attack"
MERGED_DATA_ROOT = DATA_ROOT / "merged"
campaign_names = ["dollar_char"]
campaign_ids = {name: idx for idx, name in enumerate(campaign_names)}

In [4]:
normal_train = pd.read_csv(NORMAL_DATA_ROOT / "normal_2_3_4/normal_2_3_4_flows.csv", low_memory=False)
normal_test = pd.read_csv(NORMAL_DATA_ROOT / "normal_5_6_7/normal_5_6_7_flows.csv", low_memory=False)
attack_df_list = []
for campaign_name in campaign_names:
    cid = campaign_ids[campaign_name]
    attack_train_df = pd.read_csv(ATTACK_DATA_ROOT / f"{campaign_name}_train" / f"{campaign_name}_train_labeled.csv", low_memory=False)
    attack_test_df = pd.read_csv(ATTACK_DATA_ROOT / f"{campaign_name}_test" / f"{campaign_name}_test_labeled.csv", low_memory=False)
    attack_train_df["campaign_id"] = cid
    attack_test_df["campaign_id"] = cid
    attack_df_list.append((attack_train_df, attack_test_df))

In [5]:
to_be_dropped={col for col in list(normal_train.columns) + list(normal_test.columns)  if "Unnamed" in col}
normal_train=normal_train.drop(list(to_be_dropped), axis=1, errors="ignore")
normal_test=normal_test.drop(list(to_be_dropped), axis=1, errors="ignore")

In [6]:
normal_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 9441 entries, 0 to 9440
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            9441 non-null   int64  
 1   expiration_id                 9441 non-null   int64  
 2   src_ip                        9441 non-null   str    
 3   src_mac                       9441 non-null   str    
 4   src_oui                       9441 non-null   str    
 5   src_port                      9441 non-null   int64  
 6   dst_ip                        9441 non-null   str    
 7   dst_mac                       9441 non-null   str    
 8   dst_oui                       9441 non-null   str    
 9   dst_port                      9441 non-null   int64  
 10  protocol                      9441 non-null   int64  
 11  ip_version                    9441 non-null   int64  
 12  vlan_id                       9441 non-null   int64  
 13  tunnel_id     

In [7]:
normal_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 9411 entries, 0 to 9410
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            9411 non-null   int64  
 1   expiration_id                 9411 non-null   int64  
 2   src_ip                        9411 non-null   str    
 3   src_mac                       9411 non-null   str    
 4   src_oui                       9411 non-null   str    
 5   src_port                      9411 non-null   int64  
 6   dst_ip                        9411 non-null   str    
 7   dst_mac                       9411 non-null   str    
 8   dst_oui                       9411 non-null   str    
 9   dst_port                      9411 non-null   int64  
 10  protocol                      9411 non-null   int64  
 11  ip_version                    9411 non-null   int64  
 12  vlan_id                       9411 non-null   int64  
 13  tunnel_id     

In [8]:
unnamed_cols=set()
for attack_train_df, attack_test_df in attack_df_list:
    unnamed_cols.update({col for col in list(attack_train_df.columns) + list(attack_test_df.columns) if "Unnamed" in col })
to_be_dropped=["phase_idx", "phase_name", "phase_number"] + list(unnamed_cols)
print(to_be_dropped)

['phase_idx', 'phase_name', 'phase_number']


In [9]:
for i, attack_dfs in enumerate(attack_df_list):
    attack_df_list[i] = (
        attack_dfs[0].drop(to_be_dropped, axis=1, errors="ignore"),
        attack_dfs[1].drop(to_be_dropped, axis=1, errors="ignore"),
    )

In [10]:
for campaign_id, campaign_name in enumerate(campaign_names):
    print(f"train dataframe of {campaign_name}")
    attack_df_list[campaign_id][0].info()

train dataframe of dollar_char
<class 'pandas.DataFrame'>
RangeIndex: 195044 entries, 0 to 195043
Data columns (total 90 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            195044 non-null  int64  
 1   expiration_id                 195044 non-null  int64  
 2   src_ip                        195044 non-null  str    
 3   src_mac                       195044 non-null  str    
 4   src_oui                       195044 non-null  str    
 5   src_port                      195044 non-null  int64  
 6   dst_ip                        195044 non-null  str    
 7   dst_mac                       195044 non-null  str    
 8   dst_oui                       195044 non-null  str    
 9   dst_port                      195044 non-null  int64  
 10  protocol                      195044 non-null  int64  
 11  ip_version                    195044 non-null  int64  
 12  vlan_id                 

In [11]:
for campaign_id, campaign_name in enumerate(campaign_names):
    print(f"test dataframe of {campaign_name}")
    attack_df_list[campaign_id][1].info()

test dataframe of dollar_char
<class 'pandas.DataFrame'>
RangeIndex: 110276 entries, 0 to 110275
Data columns (total 90 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            110276 non-null  int64  
 1   expiration_id                 110276 non-null  int64  
 2   src_ip                        110276 non-null  str    
 3   src_mac                       110276 non-null  str    
 4   src_oui                       110276 non-null  str    
 5   src_port                      110276 non-null  int64  
 6   dst_ip                        110276 non-null  str    
 7   dst_mac                       110276 non-null  str    
 8   dst_oui                       110276 non-null  str    
 9   dst_port                      110276 non-null  int64  
 10  protocol                      110276 non-null  int64  
 11  ip_version                    110276 non-null  int64  
 12  vlan_id                  

## Convert to IPv4

In [12]:
def build_mac_to_ipv4(dfs):
    mapping = {}
    for df in dfs:
        for mac_col, ip_col in (("src_mac", "src_ip"), ("dst_mac", "dst_ip")):
            is_v4 = ~df[ip_col].astype(str).str.contains(":", regex=False)
            pairs = df.loc[is_v4, [mac_col, ip_col]].drop_duplicates()
            mapping.update(zip(pairs[mac_col], pairs[ip_col]))
    return mapping

def resolve_ipv6_with_mac(df, mappings):
    df = df.copy()
    not_found = 0
    for mac_col, ip_col in (("src_mac", "src_ip"), ("dst_mac", "dst_ip")):
        is_v6 = df[ip_col].astype(str).str.contains(":", regex=False)
        mapped = df[mac_col].map(mappings)
        not_found += int((is_v6 & mapped.isna()).sum())
        df[ip_col] = df[ip_col].where(~(is_v6 & mapped.notna()), mapped)
    return df, not_found

In [13]:
df_list=[]
for df in attack_df_list:
    df_list += [df[0],df[1]]
df_list = df_list + [normal_train, normal_test]
mappings=build_mac_to_ipv4(df_list)

In [14]:
print(mappings.keys())

dict_keys(['12:34:7e:5d:bf:ed', '12:34:7c:fe:43:ea', '12:34:66:23:dc:bd', '12:34:f3:ef:1f:0e', '12:34:3c:ae:94:28', '12:34:2f:33:35:d0', '12:34:68:e8:37:59', '12:34:93:88:12:84', '12:34:9e:6b:c3:97', '12:34:dd:19:27:75', '12:34:08:7a:17:c7', '12:34:c6:7d:c4:17', '12:34:cd:78:ac:c7', '12:34:2f:04:c4:76', '12:34:fb:02:73:4a', '12:34:19:14:0c:5b', '12:34:d2:15:14:d2', '12:34:13:7c:72:38', '12:34:88:d1:ed:07', '12:34:9b:aa:04:c0', '12:34:ce:17:98:03', '12:34:a2:17:83:65', '12:34:01:19:20:f8', '12:34:8a:36:33:9a', '12:34:95:9d:7a:78', '12:34:d1:d1:5e:f0', '12:34:c3:b2:20:80', '12:34:ae:00:21:04', '12:34:7e:c2:90:0c', '12:34:2f:bd:b2:c1', '12:34:41:50:07:7c', '12:34:73:00:54:80', '12:34:86:5d:b7:96', '12:34:30:40:e0:39', '12:34:48:bd:51:d3', '12:34:ab:ca:6b:5f', '12:34:76:cf:4c:7e', '12:34:d6:2e:4c:74', '12:34:26:0e:c5:a6', '12:34:80:b8:c1:2f', '12:34:32:bb:6b:ea', '12:34:02:8e:4e:30', '12:34:2c:68:8c:0a', '12:34:b7:47:c9:c9', '12:34:73:b3:6d:24', '12:34:d4:3b:81:a1', '12:34:6f:ad:cc:cd', '1

In [15]:
print(mappings)

{'12:34:7e:5d:bf:ed': '10.0.0.2', '12:34:7c:fe:43:ea': '10.0.0.21', '12:34:66:23:dc:bd': '10.0.0.22', '12:34:f3:ef:1f:0e': '10.0.0.23', '12:34:3c:ae:94:28': '10.0.0.20', '12:34:2f:33:35:d0': '10.0.0.19', '12:34:68:e8:37:59': '10.0.0.18', '12:34:93:88:12:84': '10.0.0.17', '12:34:9e:6b:c3:97': '10.0.0.16', '12:34:dd:19:27:75': '10.0.0.15', '12:34:08:7a:17:c7': '10.0.0.14', '12:34:c6:7d:c4:17': '10.0.0.4', '12:34:cd:78:ac:c7': '10.0.0.5', '12:34:2f:04:c4:76': '10.0.0.6', '12:34:fb:02:73:4a': '10.0.0.7', '12:34:19:14:0c:5b': '10.0.0.8', '12:34:d2:15:14:d2': '10.0.0.9', '12:34:13:7c:72:38': '10.0.0.10', '12:34:88:d1:ed:07': '10.0.0.11', '12:34:9b:aa:04:c0': '10.0.0.12', '12:34:ce:17:98:03': '10.0.0.1', '12:34:a2:17:83:65': '10.0.0.13', '12:34:01:19:20:f8': '10.0.0.2', '12:34:8a:36:33:9a': '10.0.0.12', '12:34:95:9d:7a:78': '10.0.0.21', '12:34:d1:d1:5e:f0': '10.0.0.22', '12:34:c3:b2:20:80': '10.0.0.23', '12:34:ae:00:21:04': '10.0.0.20', '12:34:7e:c2:90:0c': '10.0.0.13', '12:34:2f:bd:b2:c1': '

In [16]:
print("unique mac addresses for the normal train dataframe:")
print("dest mac:")
print(normal_train["dst_mac"].unique())
print("src mac:")
print(normal_train["src_mac"].unique())

unique mac addresses for the normal train dataframe:
dest mac:
<StringArray>
['33:33:00:00:00:16', '33:33:ff:a8:61:93', '33:33:ff:ce:9c:13',
 '33:33:00:00:00:02', '33:33:00:00:00:fb', '33:33:ff:b0:96:9f',
 '33:33:00:00:00:01', '12:34:a5:15:3c:3e', '33:33:ff:af:82:cc',
 '33:33:ff:f6:55:18', '33:33:ff:7e:a6:16', '33:33:ff:c5:96:4e',
 '33:33:ff:ee:df:90', '33:33:ff:35:65:f2', '33:33:ff:53:d6:db',
 '33:33:ff:54:62:10', '33:33:ff:53:10:d0', '33:33:ff:b7:17:bb',
 '33:33:ff:d6:c6:eb', '33:33:ff:78:c5:d1', '33:33:ff:f0:40:33',
 '33:33:ff:e7:90:05', '33:33:ff:32:5e:9f', '33:33:ff:40:02:8d',
 '33:33:ff:09:0c:2c', '33:33:ff:bf:a7:31', '33:33:ff:8e:fc:c6',
 '33:33:ff:51:37:08', '12:34:0d:f0:9c:36', '12:34:73:b3:6d:24',
 '12:34:11:9f:d4:8e', '12:34:32:ca:d7:20', '12:34:9f:d5:63:9f',
 '12:34:d4:3b:81:a1', '12:34:7f:bb:40:6c', '12:34:8f:d5:2b:9a',
 '12:34:c1:b5:04:96', '12:34:6f:ad:cc:cd', '12:34:a1:83:ca:33',
 '12:34:d7:7e:03:0b', '12:34:2e:58:19:93', '12:34:25:3a:5e:08',
 '12:34:78:e0:c0:a0', '12:3

In [17]:
print("unique mac addresses for the normal test dataframe:")
print("dest mac:")
print(normal_test["dst_mac"].unique())
print("src mac:")
print(normal_test["src_mac"].unique())

unique mac addresses for the normal test dataframe:
dest mac:
<StringArray>
['33:33:00:00:00:16', '33:33:ff:b0:96:9f', '33:33:ff:53:d6:db',
 '33:33:ff:ee:df:90', '33:33:ff:b7:17:bb', '12:34:4c:3c:6b:02',
 '33:33:ff:7e:a6:16', '33:33:00:00:00:02', '33:33:00:00:00:fb',
 '33:33:ff:32:5e:9f', '33:33:ff:8e:fc:c6', '33:33:ff:bf:a7:31',
 '33:33:ff:40:02:8d', '33:33:ff:a8:61:93', '33:33:ff:c5:96:4e',
 '33:33:ff:af:82:cc', '33:33:ff:35:65:f2', '33:33:ff:f0:40:33',
 '33:33:ff:78:c5:d1', '33:33:ff:f6:55:18', '33:33:ff:09:0c:2c',
 '33:33:ff:53:10:d0', '33:33:ff:e7:90:05', '33:33:ff:ce:9c:13',
 '33:33:ff:d6:c6:eb', '33:33:ff:54:62:10', '33:33:ff:51:37:08',
 '33:33:00:00:00:01', '12:34:b9:51:e1:bf', '12:34:ec:17:68:8e',
 '12:34:43:31:d1:bc', '12:34:98:73:9b:79', '12:34:a7:15:f0:74',
 '12:34:32:47:fc:69', '12:34:ec:c5:cc:55', '12:34:da:e9:f7:5f',
 '12:34:50:6c:ce:be', '12:34:e6:e8:7c:cc', '12:34:c3:58:c9:90',
 '12:34:1d:c5:bf:40', '12:34:c3:e8:4f:5f', '12:34:c0:72:98:90',
 '12:34:ee:3f:53:27', '12:34

In [18]:
for campaign_id, campaign_name in enumerate(campaign_names):
    print(f"unique mac addresses for the {campaign_name} train dataframe:")
    print("dst mac:")
    print(attack_df_list[campaign_id][0]["dst_mac"].unique())
    print("src mac:")
    print(attack_df_list[campaign_id][0]["src_mac"].unique())
    print()
    print(f"unique mac addresses for the {campaign_name} test dataframe:")
    print("dst mac:")
    print(attack_df_list[campaign_id][1]["dst_mac"].unique())
    print("src mac:")
    print(attack_df_list[campaign_id][1]["src_mac"].unique())

unique mac addresses for the dollar_char train dataframe:
dst mac:
<StringArray>
['33:33:00:00:00:01', '33:33:00:00:00:02', '12:34:c6:7d:c4:17',
 '12:34:2f:04:c4:76', '12:34:19:14:0c:5b', '12:34:cd:78:ac:c7',
 '12:34:fb:02:73:4a', '12:34:a2:17:83:65', '12:34:9b:aa:04:c0',
 '12:34:88:d1:ed:07', '12:34:d2:15:14:d2', '12:34:13:7c:72:38',
 '12:34:9e:6b:c3:97', '12:34:93:88:12:84', '12:34:68:e8:37:59',
 '12:34:2f:33:35:d0', '12:34:3c:ae:94:28', '12:34:7c:fe:43:ea',
 '12:34:66:23:dc:bd', '12:34:f3:ef:1f:0e', '12:34:ce:17:98:03',
 '12:34:08:7a:17:c7', '12:34:dd:19:27:75', '33:33:00:00:00:fb',
 '12:34:7e:5d:bf:ed']
Length: 25, dtype: str
src mac:
<StringArray>
['12:34:7e:5d:bf:ed', '12:34:93:88:12:84', '12:34:f3:ef:1f:0e',
 '12:34:2f:04:c4:76', '12:34:fb:02:73:4a', '12:34:c6:7d:c4:17',
 '12:34:13:7c:72:38', '12:34:88:d1:ed:07', '12:34:9b:aa:04:c0',
 'd6:c6:da:6a:f2:23', 'f2:6f:34:cc:e6:80', '12:1c:63:b7:2e:27',
 '8e:6c:4b:ed:88:52', '3a:87:65:11:e9:ef', 'de:f4:23:ef:2e:b2',
 '1a:fa:bb:36:54:a1

In [19]:
normal_train_mapped, normal_train_no_matches=resolve_ipv6_with_mac(normal_train, mappings)
normal_test_mapped, normal_test_no_matches=resolve_ipv6_with_mac(normal_test, mappings)

In [20]:
print(f"number of ipv6 addresses without mac matches found in the normal train dataframe: {normal_train_no_matches}")
print(f"number of ipv6 addresses without mac matches found in the normal test dataframe: {normal_test_no_matches}")

number of ipv6 addresses without mac matches found in the normal train dataframe: 9902
number of ipv6 addresses without mac matches found in the normal test dataframe: 9955


In [21]:
print(f"normal train dataframe: \n{normal_train_mapped.head()}")
print(f"normal test dataframe's head: \n{normal_test_mapped.head()}")

normal train dataframe: 
   id  expiration_id                     src_ip            src_mac   src_oui  \
0   0              0                         ::  96:a5:fa:53:10:d0  96:a5:fa   
1   1              0                         ::  fa:be:65:a8:61:93  fa:be:65   
2   2              0                         ::  52:1f:a5:ce:9c:13  52:1f:a5   
3   3              0                   10.0.0.4  12:34:6f:ad:cc:cd  12:34:6f   
4   4              0  fe80::4020:90ff:fe67:1afb  42:20:90:67:1a:fb  42:20:90   

   src_port             dst_ip            dst_mac   dst_oui  dst_port  ...  \
0         0           ff02::16  33:33:00:00:00:16  33:33:00         0  ...   
1         0  ff02::1:ffa8:6193  33:33:ff:a8:61:93  33:33:ff         0  ...   
2         0  ff02::1:ffce:9c13  33:33:ff:ce:9c:13  33:33:ff         0  ...   
3         0            ff02::2  33:33:00:00:00:02  33:33:00         0  ...   
4      5353           ff02::fb  33:33:00:00:00:fb  33:33:00      5353  ...   

   dst2src_fin_packets  a

In [22]:
attack_df_mapped_list=[]
attack_no_matches_list=[]
for attack_train_df, attack_test_df in attack_df_list:
    attack_train_mapped_df, attack_train_no_matches = resolve_ipv6_with_mac(attack_train_df, mappings)
    attack_test_mapped_df, attack_test_no_matches = resolve_ipv6_with_mac(attack_test_df, mappings)
    attack_df_mapped_list.append((attack_train_mapped_df, attack_test_mapped_df))
    attack_no_matches_list.append((attack_train_no_matches, attack_test_no_matches))

In [23]:
for campaign_id, campaign_name in enumerate(campaign_names):
    print(f"number of ipv6 addresses without mac matches found in the {campaign_name} train dataframe: {attack_no_matches_list[campaign_id][0]}")
    print(f"{campaign_name} train dataframe's head: \n{attack_df_mapped_list[campaign_id][0].head()}")
    print()
    print(f"number of ipv6 addresses without mac matches found in the {campaign_name} test dataframe: {attack_no_matches_list[campaign_id][1]}")
    print(f"{campaign_name} test dataframe's head: \n{attack_df_mapped_list[campaign_id][1].head()}")
    print()

number of ipv6 addresses without mac matches found in the dollar_char train dataframe: 3516
dollar_char train dataframe's head: 
    id  expiration_id     src_ip            src_mac   src_oui  src_port  \
0  318              0   10.0.0.2  12:34:7e:5d:bf:ed  12:34:7e     50784   
1  176              0  10.0.0.17  12:34:93:88:12:84  12:34:93         0   
2  177              0  10.0.0.23  12:34:f3:ef:1f:0e  12:34:f3         0   
3  319              0   10.0.0.2  12:34:7e:5d:bf:ed  12:34:7e     52164   
4  320              0   10.0.0.2  12:34:7e:5d:bf:ed  12:34:7e     36923   

    dst_ip            dst_mac   dst_oui  dst_port  ...  \
0  ff02::1  33:33:00:00:00:01  33:33:00       547  ...   
1  ff02::2  33:33:00:00:00:02  33:33:00         0  ...   
2  ff02::2  33:33:00:00:00:02  33:33:00         0  ...   
3  ff02::1  33:33:00:00:00:01  33:33:00       547  ...   
4  ff02::1  33:33:00:00:00:01  33:33:00       547  ...   

   application_confidence  requested_server_name  client_fingerprint  \

In [24]:
from preprocessing.network import filter_ipv4_flows
normal_test_ipv4 = filter_ipv4_flows(normal_test_mapped)
normal_train_ipv4 = filter_ipv4_flows(normal_train_mapped)

In [25]:
from preprocessing.network import filter_ipv4_flows
attack_df_ipv4_list = []
for attack_train_mapped_df, attack_test_mapped_df in attack_df_mapped_list:
    attack_df_ipv4_list.append((filter_ipv4_flows(attack_train_mapped_df), filter_ipv4_flows(attack_test_mapped_df)))

In [26]:
print("overview of ipv4 converted normal train dataframe:")
print(normal_train_ipv4.info())
print()
print(normal_train_ipv4["src_ip"].value_counts())
print()
print(normal_train_ipv4["dst_ip"].value_counts())
print("overview of ipv4 converted normal test dataframe:")
print(normal_test_ipv4.info())
print()
print(normal_test_ipv4["src_ip"].value_counts())
print()
print(normal_test_ipv4["dst_ip"].value_counts())

overview of ipv4 converted normal train dataframe:
<class 'pandas.DataFrame'>
Index: 443 entries, 194 to 9181
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            443 non-null    int64  
 1   expiration_id                 443 non-null    int64  
 2   src_ip                        443 non-null    str    
 3   src_mac                       443 non-null    str    
 4   src_oui                       443 non-null    str    
 5   src_port                      443 non-null    int64  
 6   dst_ip                        443 non-null    str    
 7   dst_mac                       443 non-null    str    
 8   dst_oui                       443 non-null    str    
 9   dst_port                      443 non-null    int64  
 10  protocol                      443 non-null    int64  
 11  ip_version                    443 non-null    int64  
 12  vlan_id                   

In [27]:
for campaign_id, campaign_name in enumerate(campaign_names):
    print(f"overview of ipv4 converted {campaign_name} train dataframe:")
    print(attack_df_ipv4_list[campaign_id][0].info())
    print(attack_df_ipv4_list[campaign_id][0]["src_ip"].value_counts())
    print(attack_df_ipv4_list[campaign_id][0]["dst_ip"].value_counts())
    print(f"overview of ipv4 converted {campaign_name} test dataframe:")
    print(attack_df_ipv4_list[campaign_id][1].info())
    print(attack_df_ipv4_list[campaign_id][1]["src_ip"].value_counts())
    print(attack_df_ipv4_list[campaign_id][1]["dst_ip"].value_counts())

overview of ipv4 converted dollar_char train dataframe:
<class 'pandas.DataFrame'>
Index: 191809 entries, 16 to 195035
Data columns (total 90 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            191809 non-null  int64  
 1   expiration_id                 191809 non-null  int64  
 2   src_ip                        191809 non-null  str    
 3   src_mac                       191809 non-null  str    
 4   src_oui                       191809 non-null  str    
 5   src_port                      191809 non-null  int64  
 6   dst_ip                        191809 non-null  str    
 7   dst_mac                       191809 non-null  str    
 8   dst_oui                       191809 non-null  str    
 9   dst_port                      191809 non-null  int64  
 10  protocol                      191809 non-null  int64  
 11  ip_version                    191809 non-null  int64  
 12  vla

In [28]:
for campaign_id, campaign_name in enumerate(campaign_names):
    print(f"overview of attack step composition in ipv4 converted {campaign_name} train dataframe:")
    attack_steps = (
        attack_df_ipv4_list[campaign_id][0]
        .groupby('label', as_index=False)
        .agg(total_steps=('step_number', 'nunique'))
    )
    print(attack_steps)

overview of attack step composition in ipv4 converted dollar_char train dataframe:
                   label  total_steps
0  brute_force_malformed            1
1            dollar_char          166
2             nmap_10_T5            9
3            nmap_banner           14
4              nmap_mqtt           21
5               nmap_sub            7
6               scp_inst            7


In [29]:
for campaign_id, campaign_name in enumerate(campaign_names):
    print(f"overview of attack step composition in ipv4 converted {campaign_name} test dataframe:")
    attack_steps = (
        attack_df_ipv4_list[campaign_id][1]
        .groupby('label', as_index=False)
        .agg(total_steps=('step_number', 'nunique'))
    )
    print(attack_steps)

overview of attack step composition in ipv4 converted dollar_char test dataframe:
                   label  total_steps
0  brute_force_malformed            1
1            dollar_char           63
2               mqtt_cat            1
3             nmap_10_T5            5
4            nmap_banner           20
5              nmap_mqtt           29
6               scp_inst            9


## Augment normal dataset

In [30]:
from preprocessing.augmentation import EXCLUDED_FEATURE_KEYWORDS
keywords_excluded = EXCLUDED_FEATURE_KEYWORDS
N_REPS = 500
np.random.seed(42)

In [31]:
from preprocessing.augmentation import augment_normal_flows

def generate_dataframe(df, start_time, repetitions):
    return augment_normal_flows(df, start_time, repetitions, excluded_keywords=keywords_excluded)

In [32]:
normal_train_dfs = [normal_train_ipv4]
for i in range(1, N_REPS):
    normal_train_dfs.append(
        generate_dataframe(
            normal_train_dfs[0],
            normal_train_dfs[i - 1]["bidirectional_last_seen_ms"].max(),
            N_REPS,
        )
    )

In [33]:
normal_train_aug = pd.concat(normal_train_dfs, ignore_index=True)
normal_train_aug.info()

<class 'pandas.DataFrame'>
RangeIndex: 221500 entries, 0 to 221499
Data columns (total 86 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            221500 non-null  int64  
 1   expiration_id                 221500 non-null  int64  
 2   src_ip                        221500 non-null  str    
 3   src_mac                       221500 non-null  str    
 4   src_oui                       221500 non-null  str    
 5   src_port                      221500 non-null  int64  
 6   dst_ip                        221500 non-null  str    
 7   dst_mac                       221500 non-null  str    
 8   dst_oui                       221500 non-null  str    
 9   dst_port                      221500 non-null  int64  
 10  protocol                      221500 non-null  object 
 11  ip_version                    221500 non-null  int64  
 12  vlan_id                       221500 non-null  int64  


In [34]:
normal_train_aug.head()

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,dst2src_fin_packets,application_name,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type
0,194,1,10.0.0.22,12:34:73:b3:6d:24,12:34:73,34863,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,0,MQTT,RPC,0,6,NaN,NaN,NaN,NaN,NaN
1,201,1,10.0.0.23,12:34:d4:3b:81:a1,12:34:d4,50323,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,0,MQTT,RPC,0,6,NaN,NaN,NaN,NaN,NaN
2,246,0,10.0.0.4,12:34:6f:ad:cc:cd,12:34:6f,57437,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,1,MQTT,RPC,1,1,NaN,NaN,NaN,NaN,NaN
3,248,0,10.0.0.5,12:34:d7:7e:03:0b,12:34:d7,53963,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,1,MQTT,RPC,1,1,NaN,NaN,NaN,NaN,NaN
4,249,0,10.0.0.6,12:34:7f:bb:40:6c,12:34:7f,50217,10.0.0.1,12:34:a5:15:3c:3e,12:34:a5,1883,...,1,MQTT,RPC,1,1,NaN,NaN,NaN,NaN,NaN


In [35]:
normal_test_dfs = [normal_test_ipv4]
for i in range(1, N_REPS):
    normal_test_dfs.append(
        generate_dataframe(
            normal_test_dfs[0],
            normal_test_dfs[i - 1]["bidirectional_last_seen_ms"].max(),
            N_REPS,
        )
    )

In [36]:
normal_test_aug = pd.concat(normal_test_dfs, ignore_index=True)
normal_test_aug.info()

<class 'pandas.DataFrame'>
RangeIndex: 188000 entries, 0 to 187999
Data columns (total 86 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            188000 non-null  int64  
 1   expiration_id                 188000 non-null  int64  
 2   src_ip                        188000 non-null  str    
 3   src_mac                       188000 non-null  str    
 4   src_oui                       188000 non-null  str    
 5   src_port                      188000 non-null  int64  
 6   dst_ip                        188000 non-null  str    
 7   dst_mac                       188000 non-null  str    
 8   dst_oui                       188000 non-null  str    
 9   dst_port                      188000 non-null  int64  
 10  protocol                      188000 non-null  object 
 11  ip_version                    188000 non-null  int64  
 12  vlan_id                       188000 non-null  int64  


In [37]:
normal_test_aug.head()

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,dst2src_fin_packets,application_name,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type
0,5,0,10.0.0.7,12:34:e6:e8:7c:cc,12:34:e6,49695,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,1,MQTT,RPC,1,1,NaN,NaN,NaN,NaN,NaN
1,39,0,10.0.0.8,12:34:c3:58:c9:90,12:34:c3,37429,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,1,MQTT,RPC,1,1,NaN,NaN,NaN,NaN,NaN
2,46,0,10.0.0.4,12:34:da:e9:f7:5f,12:34:da,47577,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,1,MQTT,RPC,1,1,NaN,NaN,NaN,NaN,NaN
3,47,0,10.0.0.4,12:34:da:e9:f7:5f,12:34:da,35271,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,1,MQTT,RPC,0,6,NaN,NaN,NaN,NaN,NaN
4,51,0,10.0.0.5,12:34:fa:fb:5e:7d,12:34:fa,47799,10.0.0.1,12:34:4c:3c:6b:02,12:34:4c,1883,...,1,MQTT,RPC,1,1,NaN,NaN,NaN,NaN,NaN


## Convert to absolute time

In [38]:
from preprocessing.time import absolute_flow_time
convert_to_absolute_time = absolute_flow_time

In [39]:
from preprocessing.time import shift_capture_time

def load_start_time(npz_path):
    with np.load(npz_path, allow_pickle=True) as data:
        key = "datetime" if "datetime" in data.files else data.files[0]
        return shift_capture_time(data[key].item())

normal_train_start_time = load_start_time(NORMAL_DATA_ROOT / "normal_2_3_4" / "start_time.npz")
normal_test_start_time = load_start_time(NORMAL_DATA_ROOT / "normal_5_6_7" / "start_time.npz")

In [40]:
normal_train_aug["timestamp"] = normal_train_aug["bidirectional_first_seen_ms"].apply(
    convert_to_absolute_time, start_time=normal_train_start_time
)
normal_train_aug["campaign_id"] = -1
normal_train_aug["label"] = "normal"
normal_train_aug["step_number"] = -1

In [41]:
normal_test_aug["timestamp"] = normal_test_aug["bidirectional_first_seen_ms"].apply(
    convert_to_absolute_time, start_time=normal_test_start_time
)
normal_test_aug["campaign_id"] = -1
normal_test_aug["label"] = "normal"
normal_test_aug["step_number"] = -1

## Save augmented normal dataset

In [42]:
MERGED_DATA_ROOT.mkdir(parents=True, exist_ok=True)
normal_train_aug.to_csv(MERGED_DATA_ROOT / "normal_train_aug.csv", index=False)
normal_test_aug.to_csv(MERGED_DATA_ROOT / "normal_test_aug.csv", index=False)

## Concatenate the dataframes

In [43]:
attack_test_dfs = [attack_test_df for _, attack_test_df in attack_df_ipv4_list]
df_test = pd.concat([normal_test_aug, *attack_test_dfs], ignore_index=True)

In [ ]:
df_test.info()

In [ ]:
attack_train_dfs = [attack_train_df for attack_train_df, _ in attack_df_ipv4_list]
df_train = pd.concat([normal_train_aug, *attack_train_dfs], ignore_index=True)

In [ ]:
df_train.info()

## Save results

In [ ]:
MERGED_DATA_ROOT.mkdir(parents=True, exist_ok=True)
df_train.to_csv(MERGED_DATA_ROOT / "train.csv", index=False)
df_test.to_csv(MERGED_DATA_ROOT / "test.csv", index=False)